### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.quinex_client import QuinexClient
from quinex_application.visualisation.metadata_visualisation import VisualisationMetadata
from quinex_application.quinex_processing.mongos_store import MongoStore

In [4]:
quinex = QuinexClient()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2022_abstract_fallback")

### **Create Bulk Analysis**

In [5]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quinex_application_query4 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [ ]:
visualisation.load_df(query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df
len(df_doi)

In [ ]:
 # Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2011 = dfs_by_year[2011]
print(len(df_2011))

## **Add Papers via DOI**

In [ ]:
mongodb.insert_paper(name, df_doi, force_redownload=True, only_open_access=False)

## **Add Abstracts via CSV**

In [ ]:
df_1 = df_2011[df_2011["Abstract"].notna()]
print(len(df_1))
mongodb.insert_scopus_csv(name, df_1, force_overwrite=True)

In [ ]:
collection = mongodb.get_collection("failed_papers")
len(collection)

### **Successfully Added Papers**

In [ ]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_wos_abstract_fallback", True)
print(len(dois_df))

## **Perform Bulk Analysis**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=6)

## **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)

## **Update Paper Parquet**

In [ ]:
df = mongodb.update_paper_parquet("papers_wos_abstract_fallback", "../data/paper/paper_dataset.parquet", "24.11.2025", "17.04.2025", abstract=True)

In [1]:
import pandas as pd
df = pd.read_parquet("../data/paper/paper_dataset.parquet")

In [2]:
print("-------------------QUINEX Performed-------------------------------")
print(len(df[(df["Quinex"] == True)]))
print("-------------------QUINEX Tried-----------------------------------")
print(len(df[(df["Download_Status"] == "Succeeded")]) +  len(df[(df["Download_Status"] == "Abstract Succeeded")]) + len(df[(df["Download_Status"] == "Failed")]))
print("-------------------Overall Papers-------------------------------")
print(len(df[
    (
        (df['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True)
        | 
        (df['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == True)
        | 
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)
        |
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025'] == True)
    )
    & 
    (df['Year'] >= 2010)
]))

-------------------QUINEX Performed-------------------------------
57523
-------------------QUINEX Tried-----------------------------------
76306
-------------------Overall Papers-------------------------------
76306


In [3]:
print(len(df[(df["Download_Status"] == "Abstract Succeeded")]))
print(len(df[(df["Download_Status"] == "Succeeded")]))
print(len(df[(df["Download_Status"] == "Failed")]))

44418
31636
252
